### **Importation**

In [ ]:
# Utilitaires de base
import builtins
import pandas as pd

# Suivi des expriences (MLflow & DagsHub)
import dagshub
import mlflow

# Scikit-Learn : Sparation des donnes et mtriques d'valuation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    f1_score, 
    fbeta_score, 
    precision_score, 
    recall_score
)

# TensorFlow / Keras : Cration et entranement du modle Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import TextVectorization

#### **DagsHub & MLflow Init**

In [ ]:
# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.

_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


Initialized MLflow to track repo "Oscar-AS/disaster-tweets-project"

Repository Oscar-AS/disaster-tweets-project initialized!

MLflow activé avec succès sur DagsHub !


#### Importation Données

In [ ]:
# Chargement des données
df = pd.read_csv("Base/tweets_clean.csv")

### **Séparation des données**

In [ ]:

# Séparation Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    df, df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# Affiche dans la console le nombre de tweets utilisés pour l'entraînement
print(f"Taille de l'entraînement : {len(X_train)}")
# Affiche dans la console le nombre de tweets gardés pour le test
print(f"Taille du test : {len(X_test)}")


#### **Implémentation des modèles (Réseaux de Neurones avec TensorFlow/Keras)**

In [ ]:

# Définition de la taille maximale du vocabulaire autorisé (les 15000 mots les plus fréquents)
MAX_VOCAB_SIZE = 15000
# Définition de la taille maximale d'une phrase (tronquée si plus longue, remplie par des 0 si plus courte)
MAX_SEQUENCE_LENGTH = 128

# Instanciation de la couche de Vectorisation
vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE, # Limite du vocabulaire
    output_mode='int',         # Chaque mot sera remplacé par un nombre entier
    output_sequence_length=MAX_SEQUENCE_LENGTH # Fixe la longueur de toutes les séquences à 128
)

# Apprentissage du vocabulaire : on lit le texte d'entraînement pour créer le dictionnaire mot -> entier
vectorizer.adapt(X_train.to_numpy())

# Fonction utilitaire pour préparer les données afin que TensorFlow s'entraîne plus vite
def prepare_tf_dataset(X, y, batch_size=32):
    # Création d'un dataset TensorFlow à partir de nos listes Python (X et y)
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    # Groupement des données en paquets (batches) de 32, et mise en mémoire cache dynamique (AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    # Retourne le dataset optimisé
    return dataset

# Création du Dataset d'entraînement accéléré
train_ds = prepare_tf_dataset(X_train, y_train)
# Création du Dataset de test accéléré
test_ds = prepare_tf_dataset(X_test, y_test)

# Importation du module MLflow dédié à TensorFlow
import mlflow.tensorflow
# Activation du suivi automatique (enregistrera la loss, les paramètres et les modèles à chaque epoch sans coder manuellement)
mlflow.tensorflow.autolog(log_models=True)


#### **Modèle BiLSTM + GRU avec Poids de Classe**

##### **Description du modèle**
C'est une version sous stéroïdes du LSTM, combinant plusieurs architectures récurrentes.

##### **Explication du fonctionnement**
1. **BiDirectional :** Au lieu de lire la phrase uniquement de gauche à droite, la couche BiLSTM la lit *aussi* de droite à gauche en même temps. Cela permet de comprendre qu'un mot a été influencé par la fin de la phrase.
2. **Couche GRU :** Un parent plus moderne et plus léger du LSTM, ajouté ici pour extraire une dernière série d'informations.
3. **Class Weights :** Nous modifions l'équation d'entraînement pour que le modèle soit puni sévèrement s'il rate un tweet de "Désastre" (classe 1).



In [19]:
# Calcul mathématique pour compenser le déséquilibre des classes
# Récupération du nombre total d'exemples d'entraînement
total = len(y_train)
# Comptage du nombre d'exemples positifs (les 1, donc les vrais désastres)
pos = sum(y_train)
# Comptage du nombre d'exemples négatifs (les 0)
neg = total - pos

# Poids pour la classe 0 : un petit nombre (car la classe est majoritaire)
weight_for_0 = (1 / neg) * (total / 2.0)
# Poids pour la classe 1 : un grand nombre (car la classe est minoritaire)
weight_for_1 = (1 / pos) * (total / 2.0)
# Dictionnaire regroupant ces poids pour TensorFlow
class_weights = {0: weight_for_0, 1: weight_for_1}

# Lancement du Run MLflow pour le modèle optimisé
with mlflow.start_run(run_name="3.3_BiLSTM_Optimized"):
    # Architecture avancée du réseau
    model_opt = models.Sequential([
        # Vectorisation
        vectorizer,
        # Embedding plus puissant (128 dimensions au lieu de 64). mask_zero=True ignore le padding vide.
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, mask_zero=True),
        # La couche Bidirectional enveloppe le LSTM. 'return_sequences=True' permet d'enchaîner avec le GRU
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        # Dropout de 30% pour forcer le modèle à généraliser
        layers.Dropout(0.3),
        # Couche GRU, qui est un RNN plus rapide
        layers.GRU(32),
        # Une couche Dense classique pour extraire les conclusions
        layers.Dense(32, activation='relu'),
        # Un dernier Dropout avant la sortie
        layers.Dropout(0.3),
        # Sortie binaire
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Utilisation d'un "Learning Rate" (taux d'apprentissage) personnalisé, plus faible que celui par défaut
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    # Compilation
    model_opt.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    # Entraînement sur 5 epochs, EN INCLUANT l'argument 'class_weight' défini plus haut
    model_opt.fit(train_ds, validation_data=test_ds, epochs=5, class_weight=class_weights)
    
    # Transformation des probabilités en entiers 0/1
    y_pred_opt = (model_opt.predict(test_ds) > 0.5).astype(int)
    
    # Affichage
    print("\n--- Rapport BiLSTM Optimisé ---")
    print(classification_report(y_test, y_pred_opt))
    
    # Suivi explicite des métriques de test dans MLflow
    mlflow.log_metric("eval_f1_macro", f1_score(y_test, y_pred_opt, average='macro'))
    mlflow.log_metric("eval_f2_score", fbeta_score(y_test, y_pred_opt, beta=2, average='macro'))
    
    precision_cls = precision_score(y_test, y_pred_opt, average=None)
    recall_cls = recall_score(y_test, y_pred_opt, average=None)
    mlflow.log_metric("eval_precision_class_0", precision_cls[0])
    mlflow.log_metric("eval_precision_class_1", precision_cls[1])
    mlflow.log_metric("eval_recall_class_0", recall_cls[0])
    mlflow.log_metric("eval_recall_class_1", recall_cls[1])
    mlflow.log_metric("eval_accuracy", accuracy_score(y_test, y_pred_opt))
    
    # Enregistrement explicite du modèle pour la mise en production
    try:
        mlflow.tensorflow.log_model(model_opt, "model")
    except Exception as e:
        print("Erreur lors de la sauvegarde du modèle Keras :", e)


2026/05/06 11:52:20 WARNING mlflow.tensorflow: Encountered unexpected error while inferring batch size from training dataset: Sequential model 'sequential_8' has no defined input shape yet.
2026/05/06 11:52:20 WARNING mlflow.tensorflow: Failed to log training dataset information to MLflow Tracking. Reason: 'ascii' codec can't decode byte 0xe2 in position 120: ordinal not in range(128)


Epoch 1/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.6722 - loss: 0.6883

285/285 ━━━━━━━━━━━━━━━━━━━━ 119s 393ms/step - accuracy: 0.7503 - loss: 0.6674 - val_accuracy: 0.8083 - val_loss: 0.4622
Epoch 2/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.7953 - loss: 0.4816

285/285 ━━━━━━━━━━━━━━━━━━━━ 70s 245ms/step - accuracy: 0.8301 - loss: 0.4321 - val_accuracy: 0.8654 - val_loss: 0.3774
Epoch 3/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.9044 - loss: 0.3046

285/285 ━━━━━━━━━━━━━━━━━━━━ 73s 257ms/step - accuracy: 0.9119 - loss: 0.2747 - val_accuracy: 0.8799 - val_loss: 0.3173
Epoch 4/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 37s 128ms/step - accuracy: 0.9542 - loss: 0.1595 - val_accuracy: 0.8795 - val_loss: 0.3668
Epoch 5/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 35s 124ms/step - accuracy: 0.9808 - loss: 0.0847 - val_accuracy: 0.8804 - val_loss: 0.4513


2026/05/06 11:57:56 WARNING mlflow.tensorflow: Failed to infer model signature: Invalid dtype: object
2026/05/06 11:57:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:58:01 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:58:12 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmpilltexuj\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


72/72 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step

--- Rapport BiLSTM Optimisé ---
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      1851
           1       0.73      0.57      0.64       423

    accuracy                           0.88      2274
   macro avg       0.82      0.76      0.78      2274
weighted avg       0.87      0.88      0.87      2274



2026/05/06 11:59:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:59:12 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:59:24 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmpngy7uth_\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


🏃 View run 3.3_BiLSTM_Optimized at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2/runs/f1fa2aaa54b544ea98f27a859dd2a47a
🧪 View experiment at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2
